In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

# --- PATH SETUP ---
OUTPUT_DIR = Path("output").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def combine_and_clean_hcris():
    print("Loading extracted files...")
    hcris = pd.read_csv(OUTPUT_DIR / 'HCRIS_Data_v2010.txt', sep='\t', dtype={'provider_number': str})

    # Parse Dates
    date_cols = ['fy_end', 'fy_start', 'date_processed', 'date_created']
    for col in date_cols:
        hcris[col] = pd.to_datetime(hcris[col], format='mixed', dayfirst=False)

    # Clean numeric anomalies
    hcris['tot_discounts'] = hcris['tot_discounts'].abs()
    hcris['hrrp_payment'] = hcris['hrrp_payment'].abs()

    # Create fiscal year and sort
    hcris['fyear'] = hcris['fy_end'].dt.year
    hcris = hcris.drop(columns=['year']).sort_values(['provider_number', 'fyear'])

    # Handle Duplicates
    hcris['total_reports'] = hcris.groupby(['provider_number', 'fyear'])['provider_number'].transform('count')
    hcris['time_diff'] = (hcris['fy_end'] - hcris['fy_start']).dt.days

    # --- Set 1: Unique Reports ---
    unique1 = hcris[hcris['total_reports'] == 1].copy()
    unique1['source'] = 'unique reports'

    # --- Handle Remaining Duplicates ---
    dupes = hcris[hcris['total_reports'] > 1].copy()
    dupes['total_days'] = dupes.groupby(['provider_number', 'fyear'])['time_diff'].transform('sum')

    # --- Set 2: Multiple short reports ---
    dupes_short = dupes[dupes['total_days'] < 370].copy()

    sum_cols = [
        'tot_charges', 'tot_discounts', 'tot_operating_exp', 'ip_charges',
        'icu_charges', 'ancillary_charges', 'tot_discharges', 'mcare_discharges',
        'mcaid_discharges', 'tot_mcare_payment', 'secondary_mcare_payment',
        'hvbp_payment', 'hrrp_payment',
        'net_pat_rev', 'bad_debt', 'tot_uncomp_care_charges',
        'tot_uncomp_care_partial_pmts', 'cash', 'fixed_assets',
        'depr_land', 'depr_bldg', 'depr_lease', 'depr_fixed_equip',
        'depr_auto', 'depr_major_equip', 'depr_minor_equip', 'depr_HIT',
        'current_assets', 'current_liabilities',
    ]

    agg_dict = {c: 'sum' for c in sum_cols}
    agg_dict.update({
        'beds': 'max', 'fy_start': 'min', 'fy_end': 'max',
        'date_processed': 'max', 'date_created': 'min',
        'cost_to_charge': 'mean', 'new_cap_ass': 'max',
        'npi': 'first', 'report': 'first', 'status': 'first',
    })

    for c in ['street', 'city', 'state', 'zip', 'county', 'name']:
        agg_dict[c] = 'first'

    unique2 = dupes_short.groupby(['provider_number', 'fyear'], as_index=False).agg(agg_dict)
    unique2['source'] = 'total for year'

    # --- Set 3: Long reports ---
    dupes_long = dupes[dupes['total_days'] >= 370].copy()
    dupes_long['max_days'] = dupes_long.groupby(['provider_number', 'fyear'])['time_diff'].transform('max')

    mask3 = (dupes_long['max_days'] == dupes_long['time_diff']) & (dupes_long['time_diff'] >= 365)
    unique3 = dupes_long[mask3].copy()
    unique3['source'] = 'primary report'

    # --- Set 4: Weighted Average ---
    dupes_remain = dupes_long.merge(unique3[['provider_number', 'fyear']], on=['provider_number', 'fyear'],
                                    how='left', indicator=True)
    dupes_remain = dupes_remain[dupes_remain['_merge'] == 'left_only'].copy()

    for c in sum_cols:
        dupes_remain[c] = dupes_remain[c] * (dupes_remain['time_diff'] / dupes_remain['total_days'])

    unique4 = dupes_remain.groupby(['provider_number', 'fyear'], as_index=False).agg(agg_dict)
    unique4['source'] = 'weighted_average'

    # Final Combination and Save
    cols_to_keep = unique1.columns.intersection(unique2.columns)
    final = pd.concat([unique1[cols_to_keep], unique2[cols_to_keep],
                       unique3[cols_to_keep], unique4[cols_to_keep]], ignore_index=True)

    final = final.rename(columns={'fyear': 'year'}).sort_values(['provider_number', 'year'])

    # Save final year-by-year files
    for y in sorted(final['year'].unique()):
        if pd.notna(y):
            final[final['year'] == y].to_csv(OUTPUT_DIR / f'data-{int(y)}.csv', index=False)

    print(f"Cleaned data saved to {OUTPUT_DIR}. Total rows: {len(final)}")
    return final

if __name__ == "__main__":
    combine_and_clean_hcris()

Loading extracted files...
Cleaned data saved to /home/slkou/econ470/a0/work/HW5/data/output. Total rows: 84338


In [3]:
df = pd.read_csv(OUTPUT_DIR / 'HCRIS_Data_v2010.txt', sep='\t')
print(df['bad_debt'].describe())
print(df['bad_debt'].notna().sum())

count    5.982600e+04
mean     1.150211e+07
std      8.702924e+07
min     -4.144346e+07
25%      8.974572e+05
50%      3.755881e+06
75%      1.196967e+07
max      2.039768e+10
Name: bad_debt, dtype: float64
59826
